# ==============================================================================
# 💳 INSTITUTIONAL LENDING: CREDIT RISK VIA ENTERPRISE VOTING CLASSIFIER
# ==============================================================================
### Core Theoretical Principles:
Underwriting ensemble across 32,581 loan applications combining:
1. **RandomForestClassifier** (Variance reduction via deep bagged trees)
2. **GradientBoostingClassifier** (Bias reduction via sequential residual fitting)
3. **HistGradientBoostingClassifier** (Histogram-based fast binned splits)


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Credit Risk Environment ready.")


✅ STEP 1: Credit Risk Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Credit Risk Dataset (32,581 loan records).


In [2]:
# STEP 2 & 3: INGESTION & AUDIT
data_path = 'data/credit_risk/credit_risk_dataset.csv'
df = pd.read_csv(data_path)

print(f"📊 Dataset Shape: {df.shape[0]:,} loan applications, {df.shape[1]} attributes")
print(f"Default Status Distribution:\n{df['loan_status'].value_counts(normalize=True).round(3)}")
df.head(3)


📊 Dataset Shape: 32,581 loan applications, 12 attributes
Default Status Distribution:
loan_status
0    0.782
1    0.218
Name: proportion, dtype: float64


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating financial features and target `loan_status`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['loan_status']).copy()
y = df['loan_status'].copy()

print(f"Underwriting Features: {X.columns.tolist()}")


Underwriting Features: ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train Loan Portfolio: {X_train.shape[0]:,} applications")
print(f"Test Loan Portfolio : {X_test.shape[0]:,} applications")


Train Loan Portfolio: 26,064 applications
Test Loan Portfolio : 6,517 applications


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Robust imputer and encoder for institutional underwriting features.


In [5]:
# STEP 7: PREPROCESSOR
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
print("✅ STEP 7: Institutional Preprocessor configured.")


✅ STEP 7: Institutional Preprocessor configured.


/tmp/ipykernel_702334/3293230012.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


## ⚠️ STEP 8: ENTERPRISE TRI-MODEL ENSEMBLE BASELINE
Combining RF + GBM + HistGBM.


In [6]:
# STEP 8: ENSEMBLE FIT
voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)),
        ('gbm', GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=4, subsample=0.8, random_state=42)),
        ('hgbm', HistGradientBoostingClassifier(max_iter=100, learning_rate=0.08, random_state=42))
    ],
    voting='soft',
    n_jobs=-1
)

pipe_base = Pipeline([
    ('prep', preprocessor),
    ('vote', voting_clf)
])
pipe_base.fit(X_train, y_train)

roc_base = roc_auc_score(y_test, pipe_base.predict_proba(X_test)[:, 1])
print(f"Enterprise Tri-Model Voting ROC-AUC: {roc_base*100:.2f}%")


Enterprise Tri-Model Voting ROC-AUC: 93.37%


## 🎯 STEP 9: TUNING ENSEMBLE WEIGHTS VIA 3-FOLD CV
Optimizing weights among the three high-performance tree ensembles.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'vote__weights': [
        [1, 1, 1],
        [1, 2, 1],
        [1, 2, 2]
    ]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_base, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Credit Risk Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Credit Risk Hyperparameters: {'vote__weights': [1, 2, 2]}
🎯 Best 3-Fold CV ROC-AUC: 93.53%


## 📊 STEP 10 & 11: CREDIT RISK METRICS & RESIDUALS
Evaluating default detection accuracy and ROC-AUC.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("📋 INSTITUTIONAL CREDIT UNDERWRITING REPORT:")
print(classification_report(y_test, y_pred, target_names=['Performing Loan', 'Defaulted Loan']))
print(f"Test ROC-AUC Score : {roc_auc_score(y_test, y_prob):.4f}")
print(f"Test F1-Score      : {f1_score(y_test, y_pred):.4f}")


📋 INSTITUTIONAL CREDIT UNDERWRITING REPORT:
                 precision    recall  f1-score   support

Performing Loan       0.92      0.99      0.96      5095
 Defaulted Loan       0.97      0.71      0.82      1422

       accuracy                           0.93      6517
      macro avg       0.95      0.85      0.89      6517
   weighted avg       0.94      0.93      0.93      6517

Test ROC-AUC Score : 0.9371
Test F1-Score      : 0.8218


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & UNDERWRITING SMOKE TEST
Deploying pipeline and validating automated loan decisioning latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/credit_risk_voting_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_loan = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_decision = loaded_pipe.predict(sample_loan)[0]
pred_default_prob = loaded_pipe.predict_proba(sample_loan)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n💳 LIVE REAL-TIME CREDIT UNDERWRITING COMMITTEE:")
print(f"   Committee Verdict   : {'🚫 LOAN DECLINED (HIGH RISK)' if pred_decision == 1 else '✅ LOAN APPROVED (PRIME)'}")
print(f"   Default Probability : {pred_default_prob * 100:.2f}%")
print(f"   Underwriting Latency: {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/credit_risk_voting_pipeline.joblib (542,533 bytes)



💳 LIVE REAL-TIME CREDIT UNDERWRITING COMMITTEE:
   Committee Verdict   : ✅ LOAN APPROVED (PRIME)
   Default Probability : 9.12%
   Underwriting Latency: 69.194 ms (SLA < 2.0ms: CHECK)
